# OpenCV ArUco - Wizja Komputerowa i Augmented Reality

Projekt demonstrujący wykorzystanie markerów ArUco (Augmented Reality Quick Response) do:
- Generowania unikalnych kodów wizualnych
- Kalibracji kamery za pomocą szachownicy wzorcowej
- Detekcji i śledzenia markerów w czasie rzeczywistym
- Estymacji pozycji i orientacji w przestrzeni 3D
**Wymagane biblioteki**:
- `cv2` (OpenCV) - przetwarzanie obrazu
- `numpy` - operacje numeryczne
- `glob` - przeszukiwanie plików

In [1]:
import cv2
import numpy as np
import os
import glob

## 1. Generator Markerów ArUco

Markery ArUco to kwadratowe kody wizualne zawierające unikatowe identyfikatory binarne. Są używane do:
- Detekcji pozycji i orientacji w przestrzeni 3D
- Augmented Reality (AR)
- Robotyki i wizji komputerowej

**Słownik ArUco**: `cv2.aruco.DICT_4X4_100` oznacza:
- **4x4** - rozmiar markera w bitach (16 bitów na marker)
- **100** - maksymalnie 100 unikalnych markerów w słowniku

**Funkcje**:
- `cv2.aruco.getPredefinedDictionary()` - pobiera predefiniowany słownik markerów
- `cv2.aruco.generateImageMarker()` - generuje obraz markera o podanym ID
- `cv2.imwrite()` - zapisuje obraz do pliku
- `cv2.imshow()` - wyświetla obraz

In [2]:
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_100)

marker_id = 13
marker_size = 300
marker_image = cv2.aruco.generateImageMarker(aruco_dict, marker_id, marker_size)

cv2.imwrite(f"Markers/marker_{marker_id}.png", marker_image)
cv2.imshow(f"marker_{marker_id}",marker_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 2. Kalibracja Kamery

Kalibracja kamery to proces określenia parametrów wewnętrznych kamery (macierz kamery, współczynniki dystorsji). 
Pozwala na dokładną konwersję między współrzędnymi 3D a 2D.

**Parametry kalibracji**:
- **Macierz kamery** - zawiera ogniskową i punkt główny
- **Współczynniki dystorsji** - korygują zniekształcenia optyczne

**Kroki procesu**:
1. Przygotowanie szachownicy wzorcowej (9x6 pól, rozmiar pola 0.01m)
2. Zarejestrowanie punktów 3D (objpoints) i odpowiadających im punktów 2D (imgpoints)
3. Uruchomienie funkcji `cv2.calibrateCamera()` na wielu obrazach
4. Ocena błędu kalibracji (średni błąd < 0.5 px to dobry wynik)
5. Zapisanie macierzy kamery i współczynników dystorsji

**Kluczowe funkcje**:
- `cv2.findChessboardCorners()` - znajduje narożniki szachownicy w obrazie
- `cv2.cornerSubPix()` - uściśla pozycję narożników z dokładnością do podpiksela
- `cv2.calibrateCamera()` - oblicza parametry kamery na podstawie wielu obrazów
- `cv2.projectPoints()` - oblicza błąd poprzez rzutowanie punktów 3D na obraz 2D

In [5]:
CHECKERBOARD = (9, 6)
SQUARE_SIZE = 0.01

criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    30,
    0.001
)

objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[
    0:CHECKERBOARD[0],
    0:CHECKERBOARD[1]
].T.reshape(-1, 2)
objp *= SQUARE_SIZE

objpoints = []   
imgpoints = []   

IMAGES_DIR = os.path.join("../Calibration", "Images")

images = glob.glob(os.path.join(IMAGES_DIR, "*.jpg"))


print(f"Detected images: {len(images)}")
if len(images) == 0:
    raise RuntimeError("Images not found")

image_size = None

for fname in images:
    img = cv2.imread(fname)
    if img is None:
        print(f"Couldn't find: {fname}")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if image_size is None:
        image_size = gray.shape[::-1]

    ret, corners = cv2.findChessboardCorners(
        gray,
        CHECKERBOARD,
        cv2.CALIB_CB_ADAPTIVE_THRESH +
        cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    print(f"{os.path.basename(fname)} -> {'OK' if ret else 'ERROR'}")

    if ret:
        objpoints.append(objp)

        corners2 = cv2.cornerSubPix(
            gray,
            corners,
            (11, 11),
            (-1, -1),
            criteria
        )
        imgpoints.append(corners2)

        cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)

    cv2.imshow("Calibration", img)
    cv2.waitKey(300)

cv2.destroyAllWindows()

if len(objpoints) == 0:
    raise RuntimeError("No Chessboards found")

ret, cameraMatrix, distCoeffs, rvecs, tvecs = cv2.calibrateCamera(
    objpoints,
    imgpoints,
    image_size,
    None,
    None
)

print("\n=== Error per image ===")

total_error = 0
errors = []

for i in range(len(objpoints)):
    imgpoints2, _ = cv2.projectPoints(
        objpoints[i],
        rvecs[i],
        tvecs[i],
        cameraMatrix,
        distCoeffs
    )

    error = cv2.norm(imgpoints[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    errors.append(error)
    total_error += error

    print(f"Image {i:02d} | error = {error:.3f} px")

mean_error = total_error / len(objpoints)
print(f"\nAVG error: {mean_error:.3f} px")

print("\n=== Calibration results ===")
print("RMS error (OK<0.5):", ret)
print("Camera matrix:\n", cameraMatrix)
print("Distortion coefficients:\n", distCoeffs)

print("Saving to:", os.path.join("../Calibration", "camera_matrix.npy"))

np.save(os.path.join("../Calibration", "camera_matrix.npy"), cameraMatrix)
np.save(os.path.join("../Calibration", "dist_coeffs.npy"), distCoeffs)


Detected images: 21
WIN_20260118_19_45_44_Pro.jpg -> OK
WIN_20260118_19_45_51_Pro.jpg -> OK
WIN_20260118_19_45_57_Pro.jpg -> OK
WIN_20260118_19_46_07_Pro.jpg -> OK
WIN_20260118_19_51_10_Pro.jpg -> OK
WIN_20260118_19_51_17_Pro.jpg -> OK
WIN_20260118_19_51_28_Pro.jpg -> OK
WIN_20260118_19_51_36_Pro.jpg -> OK
WIN_20260118_19_51_48_Pro.jpg -> OK
WIN_20260118_19_51_58_Pro.jpg -> OK
WIN_20260118_19_52_09_Pro.jpg -> OK
WIN_20260118_19_52_14_Pro.jpg -> OK
WIN_20260118_19_52_21_Pro.jpg -> OK
WIN_20260118_19_52_54_Pro.jpg -> OK
WIN_20260118_19_53_00_Pro.jpg -> OK
WIN_20260118_19_53_07_Pro.jpg -> OK
WIN_20260118_19_53_18_Pro.jpg -> OK
WIN_20260118_19_53_40_Pro.jpg -> OK
WIN_20260118_19_56_46_Pro.jpg -> OK
WIN_20260118_19_59_09_Pro.jpg -> OK
WIN_20260118_20_02_10_Pro.jpg -> OK

=== Error per image ===
Image 00 | error = 0.060 px
Image 01 | error = 0.036 px
Image 02 | error = 0.076 px
Image 03 | error = 0.095 px
Image 04 | error = 0.071 px
Image 05 | error = 0.046 px
Image 06 | error = 0.052 px
Ima

## 3. Detekcja Markerów

Moduł detekcji markerów ArUco umożliwia:
- Znalezienie markerów w obrazie z kamery
- Określenie ich pozycji i orientacji w przestrzeni 3D
- Obliczenie odległości między markerami
- Wizualizację w postaci osi współrzędnych i kształtów 3D

**Kluczowe parametry**:
- `marker_length = 0.06` - rzeczywisty rozmiar markera w metrach (6 cm)
- `cameraMatrix` - macierz parametrów wewnętrznych kamery
- `distCoeffs` - współczynniki dystorsji

**Główne funkcje**:
- `cv2.aruco.ArucoDetector()` - inicjalizuje detektor markerów
- `detector.detectMarkers(gray)` - zwraca narożniki (corners), identyfikatory (ids) i odrzucone markery
- `cv2.aruco.estimatePoseSingleMarkers()` - oblicza wektory rotacji (rvec) i translacji (tvec) dla każdego markera
- `cv2.drawFrameAxes()` - rysuje osie współrzędnych (X=czerwona, Y=zielona, Z=niebieska)
- `cv2.projectPoints()` - rzutuje punkty 3D na obraz 2D (do rysowania kostki)

**Użyteczne obliczenia**:
1. **Odległość markera od kamery**: `distance = np.linalg.norm(tvec)`
2. **Odległość między dwoma markerami**: `distance = np.sqrt(dx² + dy² + dz²)` gdzie `d = pos2 - pos1`

**Funkcja `draw_cube()`**:
Rysuje 3D kostkę na markerze wykorzystując rzutowanie perspektywiczne. 
Definiuje 8 punktów kostki, rzutuje je na obraz 2D i rysuje krawędzie.

In [9]:
import cv2
import numpy as np

# Jeśli zrobiłeś kalibrację
#cameraMatrix = np.load('Calibration/camera_matrix.npy')
#distCoeffs = np.load('Calibration/dist_coeffs.npy')

# Bieda wersja bez kalibracji
cameraMatrix = np.array([[800, 0, 320],
                         [0, 800, 240],
                         [0, 0, 1]], dtype=np.float32)

distCoeffs = np.zeros((5,1), dtype=np.float32)


aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_100)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

marker_length = 0.06

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Camera error")
    exit()

def draw_cube(frame, rvec, tvec, size = marker_length):
    half = size / 2
    pts = np.float32([
        [-half, -half, 0],
        [ half, -half, 0],
        [ half,  half, 0],
        [-half,  half, 0],
        [-half, -half, size],
        [ half, -half, size],
        [ half,  half, size],
        [-half,  half, size]
    ])

    imgpts, _ = cv2.projectPoints(pts, rvec, tvec, cameraMatrix, distCoeffs)
    imgpts = np.int32(imgpts).reshape(-1,2)

    frame = cv2.drawContours(frame, [imgpts[:4]], -1, (0,255,0), 2)
    frame = cv2.drawContours(frame, [imgpts[4:]], -1, (0,0,255), 2)

    for i in range(4):
        frame = cv2.line(frame, tuple(imgpts[i]), tuple(imgpts[i+4]), (255,0,0), 2)
    return frame

while True:
    work, frame = cap.read()
    if not work:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w = frame.shape[:2]
    x = 10          
    y = h - 10          

    corners, ids, _ = detector.detectMarkers(gray)

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(frame, corners, ids)

        rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
            corners, marker_length, cameraMatrix, distCoeffs
        )

        for i, (rvec, tvec) in enumerate(zip(rvecs, tvecs)):
            marker_id = ids[i][0]
            cv2.drawFrameAxes(frame, cameraMatrix, distCoeffs, rvec, tvec, marker_length/2)

            if marker_id == 0:
                frame = draw_cube(frame, rvec, tvec, size=marker_length)
            

            # marker to camera distance
            distance_to_camera = np.linalg.norm(tvec[0])
            cv2.putText(frame,
                        f"ID {ids[i][0]} Dist: {distance_to_camera:.2f} m",
                        (10, 30 + i*30),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7, (0, 255, 0), 2)

        for rvec, tvec in zip(rvecs, tvecs):
            cv2.drawFrameAxes(frame, cameraMatrix, distCoeffs, rvec, tvec, 0.03)  # 3 cm osie

        # distance between markers
        if len(tvecs) == 2:
            pos1 = tvecs[0][0]  # [X, Y, Z] marker 1
            pos2 = tvecs[1][0]  # marker 2

            dx, dy, dz = pos2 - pos1
            distance = np.sqrt(dx**2 + dy**2 + dz**2)

            cv2.putText(frame, f"Distance: {distance:.3f} m",
            (x, y), cv2.FONT_HERSHEY_SIMPLEX,
            1, (0, 0, 255), 2)

    cv2.imshow("ArUco AR Test", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()
